In [ ]:
#pip install transformers datasets torch scikit-learn

In [ ]:
import torch
print(torch.cuda.is_available())

True


# **Imports**

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments


In [ ]:
train_path = "/content/drive/Shareddrives/NLP Task/trainV2.csv"

test_path = "/content/drive/Shareddrives/NLP Task/TestV2 - testV2.csv"

# **Load Dataset**

In [ ]:
df = pd.read_csv(train_path)

label_map = {"Non-Abusive": 0,"Abusive": 1,"abusive":1}

df["label"] = df["Class"].map(label_map)

In [ ]:
print(len(df))

3652


In [ ]:
df.isnull().sum()

,0
Text,0
Class,0
label,0


# **Preprocessing**

In [ ]:
def preprocess(text):
    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Fix HTML entity
    text = text.replace("&#39;", "'")

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Lowercase English only
    text = re.sub(r"[A-Za-z]+", lambda m: m.group(0).lower(), text)

    return text

df["clean_text"] = df["Text"].apply(preprocess)

# **Train / Validation Split**

In [ ]:
train_texts,val_texts,train_labels,val_labels = train_test_split(df["clean_text"].tolist(),df["label"].tolist(),test_size=0.1,random_state=42,stratify=df["label"])

# **Load IndicBERT**

In [ ]:
model_name = "ai4bharat/IndicBERTv2-MLM-only"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name,num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.75M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai4bharat/IndicBERTv2-MLM-only
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if y

# **Tokenization**

In [ ]:
def tokenize(texts):
    return tokenizer(texts,padding=True,truncation=True,max_length=128)

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)

# **Dataset Class**

In [ ]:
class TamilDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = TamilDataset(train_encodings, train_labels)
val_dataset = TamilDataset(val_encodings, val_labels)

# **Evaluation Metrics**

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    return {"accuracy": acc,"precision": precision,"recall": recall,"f1": f1}

# **Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./indicbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# **Trainer**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    #tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

# **Train**

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.452426,0.800546,0.829114,0.740113,0.782090
2,No log,0.442597,0.789617,0.735849,0.881356,0.802057
3,0.459404,0.418097,0.833333,0.822222,0.836158,0.829132
4,0.459404,0.433314,0.833333,0.822222,0.836158,0.829132


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=824, training_loss=0.37497075090130555, metrics={'train_runtime': 502.0252, 'train_samples_per_second': 26.182, 'train_steps_per_second': 1.641, 'total_flos': 864582927912960.0, 'train_loss': 0.37497075090130555, 'epoch': 4.0})

# **Evaluation**

In [ ]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.41827452182769775, 'eval_accuracy': 0.8333333333333334, 'eval_precision': 0.8222222222222222, 'eval_recall': 0.8361581920903954, 'eval_f1': 0.8291316526610645, 'eval_runtime': 2.6806, 'eval_samples_per_second': 136.535, 'eval_steps_per_second': 8.58, 'epoch': 4.0}


# **Test Prediction**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(250000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

In [ ]:
test_df = pd.read_csv(test_path)
test_df["clean_text"] = test_df["Text"].apply(preprocess)

test_encodings = tokenizer(
    test_df["clean_text"].tolist(),
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Move all tensors to device
test_encodings = {k: v.to(device) for k, v in test_encodings.items()}

model.eval()
with torch.no_grad():
    outputs = model(**test_encodings)
    preds = torch.argmax(outputs.logits, dim=1).tolist()

inv_label_map = {0: "Non-Abusive", 1: "Abusive",1: "abusive"}
test_df["prediction"] = [inv_label_map[p] for p in preds]

test_df[["Text","clean_text","prediction"]].to_csv("indicbert_submission.csv", index=False)

# **Validation Split Testing**

In [ ]:
val_predictions = trainer.predict(val_dataset)

y_true = val_labels
y_pred = val_predictions.predictions.argmax(axis=1)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    y_true,
    y_pred,
    target_names=["Non-Abusive", "Abusive"]
))

print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

 Non-Abusive       0.84      0.83      0.84       189
     Abusive       0.82      0.84      0.83       177

    accuracy                           0.83       366
   macro avg       0.83      0.83      0.83       366
weighted avg       0.83      0.83      0.83       366

[[157  32]
 [ 29 148]]


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)

Accuracy: 0.8333333333333334
Precision: 0.8222222222222222
Recall: 0.8361581920903954
F1: 0.8291316526610645


In [ ]:
print(val_dataset.labels)

[1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 

In [ ]:
for i in range(len(val_texts)):
  print(val_labels[i],y_pred[i],val_texts[i])
#print(val_texts)

1 1 அடியே.... மானங்கெட்டவளே.... என்ன hair ku ..... இதையெல்லாம் எதுக்கு சொல்ற... கல்யாணம் பன்னா இது எல்லாம் நடக்கும்ன்னனு தெரியாதா சம்யுக்தா....என்னமோ ஓவரா அழுது நடிக்கிற
1 1 எது இவங்க ஜெயலலிதாவா அப்போ இரண்டு குஷ்பு கால் சூப்பு ரெடி ...........
0 0 ஹலோ ஷகிலா அம்மா இந்த லூசு கிட்ட பேசாதீங்க அப்புறம் உங்களுக்கு தான் பைத்தியம் ஆகிவிடுவார் please amma
0 0 ரொம்ப பொறுமையா பேசுறீங்க .u r appreciate mam
1 1 iva 22 வயசுலயே இப்படி னா இன்னும் என்னலாம் செய்ய காத்திருக்காளோ எம்மாடியோ
0 0 பணம் ,அழகு , நிறம் இருக்கிற பெண்கள் என்ன தவறு வேண்டுமானாலும் செய்யலாம் என்பதை இந்த பேட்டி மூலமாக சொல்றீங்க ஏன் திவ்யாவிடம் கேட்ட ஒரு கேள்வி கூட இரு பெண் பிள்ளைக்கு தாயான வனித்தவிடம் கேட்க வில்லை இப்படி தொடர்ந்து திருமணம் செய்து கொண்டே போனால் பெண் குழந்தையின் எதிர்காலம் என்ன ஆகும் என்று ஏன் கேக்கல வனித்தவை பார்த்தால் பயமா ? நன்றாக படித்த திவ்யாவை வீட்டு வேலைக்கு போக சொல்ற ஏன் சூரியா , இலக்கியவிடம் இதை சொல்லல இந்த பெண்ணை இவ்வளவு இழிவு படுத்தி இருக்க வேண்டாம் கொஞ்சம் அன்பா பண்பா அறிவுரை சொல்லி இருக்கலாம் சகோதரி சகிலா
0

In [ ]:
print(len(val_labels))

366


In [ ]:
print(len(train_texts))

3286


In [ ]:
print(len(df))

3652
